In [1]:
import os
import pandas as pd

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["KERAS_BACKEND"] = "tensorflow"

In [6]:
from deepface import DeepFace

def oblicz_podobienstwo_do_postaci(sciezka_user, sciezka_postac):
    print(f"Porównuję Twoje zdjęcie z: {sciezka_postac}...")
    
    DeepFace.build_model("ArcFace")
    
    try:
        # Używamy funkcji verify dla pary zdjęć
        result = DeepFace.verify(
            img1_path = sciezka_user,
            img2_path = sciezka_postac,
            model_name = "VGG-Face",      # ArcFace daje świetne, stabilne wyniki
            detector_backend = "opencv", # Szybki detektor twarzy
            enforce_detection = True     # Wyrzuci błąd, jeśli na którymś zdjęciu nie ma twarzy
        )
        
        # Wyciągamy dystans wektorowy (dla ArcFace: 0.0 = identyczni, ~0.68 = zupełnie inni)
        dystans = result["distance"]
        
        # Przeliczamy dystans na intuicyjny procent podobieństwa dla aplikacji rozrywkowej
        # Ustalamy bezpieczny maksymalny próg różnicy na 0.70
        max_threshold = 0.70
        if dystans >= max_threshold:
            podobienstwo_procent = 0.0
        else:
            # Mapujemy dystans na procenty (im mniejszy dystans, tym bliżej 100%)
            podobienstwo_procent = (1 - (dystans / max_threshold)) * 100
            
        print("\n--- WYNIK PORÓWNANIA ---")
        print(f"Dystans matematyczny: {round(dystans, 4)}")
        print(f"Twoje podobieństwo do wybranej postaci to: {round(podobienstwo_procent, 1)}%")
        
        if podobienstwo_procent > 75:
            print("Niesamowite! Wyglądacie jak rodzeństwo.")
        elif podobienstwo_procent > 50:
            print("Widać pewne wspólne rysy twarzy!")
        else:
            print("Cóż... Moc nie jest z Wami silna, kompletnie inna twarz.")
            
    except Exception as e:
        print(f"Błąd podczas analizy: {e}")
        print("Upewnij się, że na obu zdjęciach wyraźnie widać twarz en face.")

# --- PRZYKŁAD UŻYCIA ---
# Masz lokalnie dwa zdjęcia w tym samym folderze co skrypt:
image = "/home/maciek/dev/face/is-this-hulk-hogan/ml/notebooks/2d10d121-226d-47a2-9f4b-5d72daff7ca6.jfif"
character = "image.png"

oblicz_podobienstwo_do_postaci(image, character)

Porównuję Twoje zdjęcie z: image.png...


2026-06-08 21:13:43.741419: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 411041792 exceeds 10% of free system memory.


26-06-08 21:13:46 - 🔗 vgg_face_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/vgg_face_weights.h5 to /home/maciek/.deepface/weights/vgg_face_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/vgg_face_weights.h5
To: /home/maciek/.deepface/weights/vgg_face_weights.h5
100%|██████████| 580M/580M [00:40<00:00, 14.4MB/s] 



--- WYNIK PORÓWNANIA ---
Dystans matematyczny: 0.858
Twoje podobieństwo do wybranej postaci to: 0.0%
Cóż... Moc nie jest z Wami silna, kompletnie inna twarz.


In [ ]:
from deepface import DeepFace
import numpy as np

def oblicz_podobienstwo_emocji(sciezka_user, sciezka_postac):
    # Analizuj emocje na obu zdjęciach osobno
    wynik_user = DeepFace.analyze(
        img_path=sciezka_user,
        actions=["emotion"],
        detector_backend="opencv"
    )[0]["emotion"]
    
    wynik_postac = DeepFace.analyze(
        img_path=sciezka_postac,
        actions=["emotion"],
        detector_backend="opencv"
    )[0]["emotion"]
    
    print(f"Twoje emocje:       {wynik_user}")
    print(f"Emocje postaci:     {wynik_postac}")
    
    # Porównaj rozkłady emocji - im bardziej zbliżone, tym wyższy %
    # Używamy podobieństwa cosinusowego między wektorami emocji
    emocje = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
    vec_user = np.array([wynik_user[e] for e in emocje])
    vec_postac = np.array([wynik_postac[e] for e in emocje])
    
    podobienstwo = np.dot(vec_user, vec_postac) / (
        np.linalg.norm(vec_user) * np.linalg.norm(vec_postac)
    )
    podobienstwo_procent = round(podobienstwo * 100, 1)
    
    print(f"\nPodobieństwo wyrazu twarzy: {podobienstwo_procent}%")
    return podobienstwo_procent


image = "/home/maciek/dev/face/is-this-hulk-hogan/ml/notebooks/2d10d121-226d-47a2-9f4b-5d72daff7ca6.jfif"
character = "image.png"

oblicz_podobienstwo_emocji(image, character)

26-06-08 21:17:03 - 🔗 facial_expression_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5 to /home/maciek/.deepface/weights/facial_expression_model_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: /home/maciek/.deepface/weights/facial_expression_model_weights.h5
100%|██████████| 5.98M/5.98M [00:00<00:00, 8.88MB/s]


Twoje emocje:       {'angry': 2.806827612221241, 'disgust': 4.421907107143852e-06, 'fear': 44.88860368728638, 'happy': 0.0011726642696885392, 'sad': 22.993746399879456, 'surprise': 0.013637819210998714, 'neutral': 29.29600477218628}
Emocje postaci:     {'angry': 1.044032908976078, 'disgust': 1.9961669817103456e-08, 'fear': 0.2567550400272012, 'happy': 1.3742581295161926e-06, 'sad': 10.79997718334198, 'surprise': 2.8795443540730048e-05, 'neutral': 87.89920806884766}

Podobieństwo wyrazu twarzy: 54.9%


54.9